In [1]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import os

from catboost import CatBoostClassifier


# ==========================================
# 1. Paths
# ==========================================

os.makedirs(
    "outputs/shap",
    exist_ok=True
)


# ==========================================
# 2. Load Dataset
# ==========================================

df = pd.read_csv(
    "data/market_data_final_features.csv",
    parse_dates=["Date"],
    index_col="Date"
)

df = df.dropna(
    subset=["HMM_Regime_Risk"]
).copy()


# ==========================================
# 3. Load Final Feature List
# ==========================================

feature_df = pd.read_csv(
    "data/final_model_features.csv"
)

features = (
    feature_df["Feature"]
    .tolist()
)

print(
    "Feature Count:",
    len(features)
)


# ==========================================
# 4. Recreate Test Set
# ==========================================

split = int(
    len(df) * 0.80
)

X = df[features]

X_test = X.iloc[
    split:
].copy()

y_test = df[
    "Crash_Risk"
].iloc[
    split:
].copy()

print(
    "Test Shape:",
    X_test.shape
)


# ==========================================
# 5. Load Final CatBoost Model
# ==========================================

model = CatBoostClassifier()

model.load_model(
    "market_tda_final_model.cbm"
)


# ==========================================
# 6. SHAP Explainer
# ==========================================

print(
    "\nCalculating SHAP values..."
)

explainer = shap.TreeExplainer(
    model
)

shap_values = explainer.shap_values(
    X_test
)

shap_values = np.array(
    shap_values
)

print(
    "SHAP Shape:",
    shap_values.shape
)


# ==========================================
# 7. Global SHAP Importance
# ==========================================

mean_abs_shap = np.abs(
    shap_values
).mean(
    axis=0
)

importance = pd.DataFrame({

    "Feature":
        features,

    "Mean_Abs_SHAP":
        mean_abs_shap

}).sort_values(

    "Mean_Abs_SHAP",
    ascending=False

)

print(
    "\n========== TOP 20 SHAP FEATURES =========="
)

print(
    importance
    .head(20)
    .to_string(
        index=False
    )
)


importance.to_csv(
    "data/shap_feature_importance.csv",
    index=False
)


# ==========================================
# 8. TDA Feature Importance
# ==========================================

tda_importance = importance[
    importance[
        "Feature"
    ].str.startswith(
        "TDA_"
    )
].copy()

print(
    "\n========== TDA SHAP IMPORTANCE =========="
)

print(
    tda_importance
    .to_string(
        index=False
    )
)

tda_importance.to_csv(
    "data/tda_shap_importance.csv",
    index=False
)


# ==========================================
# 9. SHAP Summary Plot
# ==========================================

plt.figure()

shap.summary_plot(

    shap_values,

    X_test,

    max_display=20,

    show=False
)

plt.tight_layout()

plt.savefig(
    "outputs/shap/shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print(
    "\nSaved SHAP summary plot."
)


# ==========================================
# 10. SHAP Bar Plot
# ==========================================

plt.figure()

shap.summary_plot(

    shap_values,

    X_test,

    plot_type="bar",

    max_display=20,

    show=False
)

plt.tight_layout()

plt.savefig(
    "outputs/shap/shap_bar.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ==========================================
# 11. Load Prediction Probabilities
# ==========================================

predictions = pd.read_csv(

    "data/final_test_predictions.csv",

    parse_dates=["Date"],

    index_col="Date"

)

# Align dates
predictions = predictions.loc[
    X_test.index
]


# ==========================================
# 12. Explain Highest-Risk Prediction
# ==========================================

highest_risk_date = (

    predictions[
        "Crash_Probability"
    ]
    .idxmax()

)

position = X_test.index.get_loc(
    highest_risk_date
)


print(
    "\nHighest Risk Date:",
    highest_risk_date
)

print(
    "Crash Probability:",
    round(
        predictions.loc[
            highest_risk_date,
            "Crash_Probability"
        ],
        4
    )
)

print(
    "Actual Crash Risk:",
    predictions.loc[
        highest_risk_date,
        "Actual"
    ]
)


# ==========================================
# 13. Local SHAP Contributions
# ==========================================

local_values = shap_values[
    position
]

local_explanation = pd.DataFrame({

    "Feature":
        features,

    "Feature_Value":
        X_test.iloc[
            position
        ].values,

    "SHAP_Value":
        local_values

})

local_explanation[
    "Abs_SHAP"
] = np.abs(

    local_explanation[
        "SHAP_Value"
    ]

)

local_explanation = (

    local_explanation

    .sort_values(
        "Abs_SHAP",
        ascending=False
    )

)


print(
    "\n========== WHY HIGH RISK? =========="
)

print(

    local_explanation

    .head(15)

    [
        [
            "Feature",
            "Feature_Value",
            "SHAP_Value"
        ]
    ]

    .to_string(
        index=False
    )

)


local_explanation.to_csv(

    "data/highest_risk_shap_explanation.csv",

    index=False

)


# ==========================================
# 14. TDA Contribution Percentage
# ==========================================

total_shap = np.abs(
    shap_values
).sum()

tda_indices = [

    i for i, feature
    in enumerate(features)

    if feature.startswith(
        "TDA_"
    )

]

tda_shap = np.abs(

    shap_values[
        :,
        tda_indices
    ]

).sum()


tda_contribution = (

    tda_shap
    /
    total_shap
    *
    100

)


print(
    "\nOverall TDA SHAP Contribution:",
    round(
        tda_contribution,
        2
    ),
    "%"
)


# ==========================================
# 15. Save Summary
# ==========================================

with open(

    "data/shap_summary.txt",

    "w"

) as f:

    f.write(
        "SHAP EXPLAINABILITY SUMMARY\n"
    )

    f.write(
        "=" * 50
    )

    f.write(
        "\n\nTop Features:\n"
    )

    f.write(

        importance

        .head(20)

        .to_string(
            index=False
        )

    )

    f.write(

        "\n\nOverall TDA Contribution: "

        + str(
            round(
                tda_contribution,
                2
            )
        )

        + "%"

    )

    f.write(

        "\n\nHighest Risk Date: "

        + str(
            highest_risk_date
        )

    )


print(
    "\nSHAP analysis completed successfully."
)

Feature Count: 29
Test Shape: (1185, 29)

Calculating SHAP values...
SHAP Shape: (1185, 29)

========== TOP 20 SHAP FEATURES ==========
                Feature  Mean_Abs_SHAP
TDA_H0_TotalPersistence       0.335515
    SP500_Volatility_10       0.282198
 TDA_H0_MeanPersistence       0.261118
              VIX_Ratio       0.233443
                    VIX       0.205706
    SP500_Volatility_20       0.198909
   NASDAQ_Volatility_20       0.163751
               VIX_MA10       0.134102
  TDA_H0_MaxPersistence       0.124438
    SP500_MA50_Distance       0.106760
  TDA_H1_MaxPersistence       0.090900
TDA_H1_TotalPersistence       0.072746
 TDA_H1_MeanPersistence       0.066820
           TDA_H1_Count       0.061916
    SP500_MA20_Distance       0.053737
      SP500_Momentum_20       0.053268
         TDA_H1_Entropy       0.043438
         OIL_Log_Return       0.039389
      SP500_Momentum_10       0.034315
        GOLD_Log_Return       0.033878

========== TDA SHAP IMPORTANCE ==========
  